# Multi-fidelity FLD regression: AR1 and NARGP (emukit + GPy)

Inputs `[ODF GSH coefficients, rho]` -> `minor` / `major` strain, one model per target. Writes `outputs/<kernel_set>/<repr>_<form>_<strategy>/<tag>/` (metrics JSON + CSV, prediction CSVs).

In [ ]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path.cwd()                   
DATA = ROOT / "data"
assert DATA.is_dir(), f"data folder not found under {ROOT}"

REPR = "L8"               # "L8" (13 coeffs) | "L12" (37 coeffs)
N_LF_TRAIN, N_HF_TRAIN = 90, 45    # training ROW budgets (LF > HF)
N_LF_RAND, N_LF_FAR = 100, 100  # LF test subsets: random / farthest textures
N_HF_SUB = 30                   # HF rand/far subset size (HF test = all non-train textures)
STRATEGY = "random"          # "random" | "local" (cluster around a seed texture)
FORMULATION = "complete"        # "complete": full 9-rho curves | "sparse": scattered rows
KERNEL_SET = "standard"         # "engineered" | "standard" (ARD Matern-5/2)
REPEAT = 0                # 0, 1, 2
SEED = 0                        # model-fitting seed
TEST_SEED = 9000 + REPEAT
TRAIN_SEED = {"random": 100, "local": 200}[STRATEGY] * 1000 + REPEAT
TAG = (f"{REPR}_lf{N_LF_TRAIN}_hf{N_HF_TRAIN}_{STRATEGY}{REPEAT}_{FORMULATION}"
       + ("" if KERNEL_SET == "engineered" else "_" + KERNEL_SET))
OUTPUTS_NAME = "outputs"
OUT_DIR = ROOT / OUTPUTS_NAME / KERNEL_SET / f"{REPR}_{FORMULATION}_{STRATEGY}" / TAG
print("config:", TAG)

RHO_ARR = np.array([-0.5, -0.3, -0.1, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
N_RHO = len(RHO_ARR)
TARGETS = ("minor", "major")

in_lo  = pd.read_csv(DATA / f"in_lo_{REPR}.csv").sort_values("index").reset_index(drop=True)
in_hi  = pd.read_csv(DATA / f"in_hi_{REPR}.csv").sort_values("index").reset_index(drop=True)
out_lo = pd.read_csv(DATA / "out_lo_fld.csv").sort_values("index").reset_index(drop=True)
out_hi = pd.read_csv(DATA / "out_hi_fld.csv").sort_values("index").reset_index(drop=True)

feat_cols = [c for c in in_lo.columns if c not in ("filename", "index")]
targ_cols = [c for c in out_lo.columns if c not in ("filename", "index")]   # minor_k, major_k interleaved

in_lo = in_lo[in_lo["index"].isin(out_lo["index"])].reset_index(drop=True)
in_hi = in_hi[in_hi["index"].isin(out_hi["index"])].reset_index(drop=True)
assert (in_lo["index"].values == out_lo["index"].values).all()
assert (in_hi["index"].values == out_hi["index"].values).all()

X_lf_all, Y_lf_all = in_lo[feat_cols].to_numpy(float), out_lo[targ_cols].to_numpy(float)
X_hf_all, Y_hf_all = in_hi[feat_cols].to_numpy(float), out_hi[targ_cols].to_numpy(float)
lf_index_vals = out_lo["index"].to_numpy()
hf_index_vals = out_hi["index"].to_numpy()

rng_test = np.random.default_rng(TEST_SEED)
rng_train = np.random.default_rng(TRAIN_SEED)
rng_hf_sub = np.random.default_rng(TEST_SEED + 500)

def nn_dist_each(A, B):
    return np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)).min(1)

# standardized GSH space (full LF pool statistics) for the sampling rules
mu_all, sd_all = X_lf_all.mean(0), X_lf_all.std(0); sd_all[sd_all < 1e-12] = 1.0
Z_lf, Z_hf = (X_lf_all - mu_all) / sd_all, (X_hf_all - mu_all) / sd_all

# training selection first, from the full pools; "local": seed texture is the only random element
seed_row = int(rng_train.choice(len(X_lf_all)))
Z_seed = Z_lf[seed_row]

def pick_train(cand, Zc, n_rows, max_tex=None):
    """n_rows training rows from cand's (texture, rho) grid.
    Returns (sorted texture pool-row ids, rows[:, (texture position, rho index)])."""
    if FORMULATION == "complete":
        n_tex = n_rows // N_RHO
        if STRATEGY == "random":
            tex = np.sort(rng_train.choice(cand, n_tex, replace=False))
        else:
            d = np.sqrt(((Zc - Z_seed) ** 2).sum(-1))
            tex = np.sort(cand[np.argsort(d)[:n_tex]])
        rows = np.array([(p, k) for p in range(n_tex) for k in range(N_RHO)], int)
        return tex, rows
    if STRATEGY == "random":
        if max_tex is not None and len(cand) > max_tex:
            cand = np.sort(rng_train.choice(cand, max_tex, replace=False))
        tex = np.sort(cand)
        assert n_rows <= len(tex) * N_RHO, "sparse/random budget exceeds the available (texture, rho) cells"
        grid = rng_train.choice(len(tex) * N_RHO, n_rows, replace=False)
        rows = np.column_stack([grid // N_RHO, grid % N_RHO])
    else:
        # nearest textures first; rho cycled over the distance rank
        d = np.sqrt(((Zc - Z_seed) ** 2).sum(-1))
        tex = cand[np.argsort(d)]
        if max_tex is not None:
            tex = tex[:max_tex]
        T = len(tex)
        assert n_rows <= T * N_RHO, "sparse/local budget exceeds the available (texture, rho) cells"
        rows = np.array([(r % T, (r % T + r // T) % N_RHO) for r in range(n_rows)], int)
    used = np.unique(rows[:, 0])
    remap = np.full(rows[:, 0].max() + 1, -1); remap[used] = np.arange(len(used))
    rows[:, 0] = remap[rows[:, 0]]
    order = np.argsort(tex[used])
    inv = np.empty(len(used), int); inv[order] = np.arange(len(used))
    rows[:, 0] = inv[rows[:, 0]]
    return np.sort(tex[used]), rows

def build_train(X_all, Y_all, tex, rows):
    XL = np.column_stack([X_all[tex][rows[:, 0]], RHO_ARR[rows[:, 1]]])
    y = {"minor": Y_all[tex][rows[:, 0], 2 * rows[:, 1]],
         "major": Y_all[tex][rows[:, 0], 2 * rows[:, 1] + 1]}
    return XL, y

lf_tex, lf_rows = pick_train(np.arange(len(X_lf_all)), Z_lf, N_LF_TRAIN)
HF_CAP = len(X_hf_all) - 2 * N_HF_SUB      # sparse HF: keep >= 2*N_HF_SUB textures for testing
hf_tex, hf_rows = pick_train(np.arange(len(X_hf_all)), Z_hf, N_HF_TRAIN, max_tex=HF_CAP)
XL_lf_tr, yL_lf_tr = build_train(X_lf_all, Y_lf_all, lf_tex, lf_rows)
XL_hf_tr, yL_hf_tr = build_train(X_hf_all, Y_hf_all, hf_tex, hf_rows)
assert len(XL_lf_tr) == N_LF_TRAIN and len(XL_hf_tr) == N_HF_TRAIN

# distance reference = LF-train U HF-train textures, standardized on that union
tr_index = np.union1d(lf_index_vals[lf_tex], hf_index_vals[hf_tex])
X_tr_obs = X_lf_all[np.isin(lf_index_vals, tr_index)]
assert len(X_tr_obs) == len(tr_index), "every HF index must exist in the LF pool"
mu_d, sd_d = X_tr_obs.mean(0), X_tr_obs.std(0); sd_d[sd_d < 1e-12] = 1.0
Z_tr_d = (X_tr_obs - mu_d) / sd_d

# test = the remainder. HF: all non-train HF textures; LF: rand U far from the LF remainder
hf_te = np.setdiff1d(np.arange(len(X_hf_all)), hf_tex)
N_HF_TEST = len(hf_te)
assert N_HF_TEST >= 2 * N_HF_SUB, "not enough HF test textures for distinct rand/far subsets"

lf_rest = np.where(~np.isin(lf_index_vals, tr_index))[0]
d_rest = nn_dist_each((X_lf_all[lf_rest] - mu_d) / sd_d, Z_tr_d)
lf_far = np.sort(lf_rest[np.argsort(d_rest)[-N_LF_FAR:]])
lf_te = np.sort(rng_test.choice(lf_rest, N_LF_RAND, replace=False))

lf_ev = np.unique(np.concatenate([lf_te, lf_far]))
N_LF_EVAL = len(lf_ev)
lf_mask_rand = np.isin(lf_ev, lf_te)
lf_mask_far = np.isin(lf_ev, lf_far)
assert len(np.intersect1d(lf_tex, lf_ev)) == 0, "LF train overlaps LF evaluation set"
assert not np.isin(lf_index_vals[lf_ev], tr_index).any(), "LF evaluation set contains a training texture"
assert len(np.intersect1d(hf_tex, hf_te)) == 0, "HF train overlaps HF test"

X_lf_te, Y_lf_te = X_lf_all[lf_ev], Y_lf_all[lf_ev]
X_hf_te, Y_hf_te = X_hf_all[hf_te], Y_hf_all[hf_te]
lf_te_index = lf_index_vals[lf_ev]
hf_te_index = hf_index_vals[hf_te]

lf_te_dist = nn_dist_each((X_lf_te - mu_d) / sd_d, Z_tr_d)
hf_te_dist = nn_dist_each((X_hf_te - mu_d) / sd_d, Z_tr_d)

hf_mask_far = np.zeros(N_HF_TEST, bool)
hf_mask_far[np.argsort(hf_te_dist)[-N_HF_SUB:]] = True
hf_mask_rand = np.zeros(N_HF_TEST, bool)
hf_mask_rand[rng_hf_sub.choice(N_HF_TEST, N_HF_SUB, replace=False)] = True

lf_masks = {"all": np.ones(N_LF_EVAL, bool), f"rand{N_LF_RAND}": lf_mask_rand,
            f"far{N_LF_FAR}": lf_mask_far}
hf_masks = {"all": np.ones(N_HF_TEST, bool), f"rand{N_HF_SUB}": hf_mask_rand,
            f"far{N_HF_SUB}": hf_mask_far}

# long format: one row per (texture, rho), rho fastest; wide: one row per texture, targ_cols order
def to_long(X):
    return np.column_stack([np.repeat(X, N_RHO, axis=0), np.tile(RHO_ARR, len(X))])

def long_to_wide(y_by_target, n):
    wide = np.empty((n, len(targ_cols)))
    wide[:, 0::2] = y_by_target["minor"].reshape(n, N_RHO)
    wide[:, 1::2] = y_by_target["major"].reshape(n, N_RHO)
    return wide

XL_lf_te, XL_hf_te = to_long(X_lf_te), to_long(X_hf_te)

# inputs standardized on the LF-train rows
x_mu, x_sd = XL_lf_tr.mean(0), XL_lf_tr.std(0)
x_sd[x_sd < 1e-12] = 1.0
scale_x = lambda X: (X - x_mu) / x_sd
XsL_lf_tr, XsL_hf_tr = scale_x(XL_lf_tr), scale_x(XL_hf_tr)
XsL_lf_te, XsL_hf_te = scale_x(XL_lf_te), scale_x(XL_hf_te)

# outputs with zero pool variance (minor strain at rho = 0) are excluded from every metric
ACTIVE = Y_lf_all.std(0) > 1e-12
MINOR = np.zeros(len(targ_cols), bool); MINOR[0::2] = True

CONFIG = {"tag": TAG, "repr": REPR, "strategy": STRATEGY, "repeat": REPEAT,
          "formulation": FORMULATION, "kernel_set": KERNEL_SET,
          "test_seed": TEST_SEED, "train_seed": TRAIN_SEED, "seed": SEED,
          "n_lf_train_rows": N_LF_TRAIN, "n_hf_train_rows": N_HF_TRAIN,
          "n_lf_train_textures": int(len(lf_tex)), "n_hf_train_textures": int(len(hf_tex)),
          "n_train_textures_union": int(len(tr_index)),
          "distance_reference": "LF_train U HF_train textures",
          "lf_train_pairs": [[int(lf_index_vals[lf_tex][p]), float(RHO_ARR[k])] for p, k in lf_rows],
          "hf_train_pairs": [[int(hf_index_vals[hf_tex][p]), float(RHO_ARR[k])] for p, k in hf_rows],
          "n_lf_rand": N_LF_RAND, "n_lf_far": N_LF_FAR, "n_lf_eval": int(N_LF_EVAL),
          "n_hf_test": int(N_HF_TEST), "n_hf_sub": N_HF_SUB, "hf_train_texture_cap": int(HF_CAP),
          "lf_test_index": lf_te_index.tolist(),
          "lf_rand_index": lf_index_vals[lf_te].tolist(),
          "lf_far_index": lf_index_vals[lf_far].tolist(),
          "hf_test_index": hf_te_index.tolist()}

print(f"train rows -- LF: {N_LF_TRAIN} over {len(lf_tex)} textures, HF: {N_HF_TRAIN} over "
      f"{len(hf_tex)} textures | LF eval: {N_LF_EVAL} textures | HF test: {N_HF_TEST} textures")

In [ ]:
if FORMULATION == "sparse" or KERNEL_SET == "standard":
    # global per-target standardization on the LF train rows
    g_mu = {t: float(yL_lf_tr[t].mean()) for t in TARGETS}
    g_sd = {t: float(max(yL_lf_tr[t].std(), 1e-12)) for t in TARGETS}
    def scale_y(t, y_flat):
        return (y_flat - g_mu[t]) / g_sd[t]
    def unscale_mean(t, ys_flat):
        return ys_flat * g_sd[t] + g_mu[t]
    def unscale_std(t, s_flat):
        return s_flat * g_sd[t]
else:
    # per-(target, rho) standardization from the LF train textures; the same LF statistics
    # are applied to the HF targets (one shared scale for the joint model)
    Y_lf_tr_wide = Y_lf_all[lf_tex]
    col_mu = {"minor": Y_lf_tr_wide[:, 0::2].mean(0), "major": Y_lf_tr_wide[:, 1::2].mean(0)}
    col_sd = {"minor": Y_lf_tr_wide[:, 0::2].std(0),  "major": Y_lf_tr_wide[:, 1::2].std(0)}
    col_sd_safe = {t: np.where(col_sd[t] < 1e-12, 1.0, col_sd[t]) for t in TARGETS}
    def scale_y(t, y_flat):
        n = len(y_flat) // N_RHO
        return (y_flat - np.tile(col_mu[t], n)) / np.tile(col_sd_safe[t], n)
    def unscale_mean(t, ys_flat):
        n = len(ys_flat) // N_RHO
        return ys_flat * np.tile(col_sd[t], n) + np.tile(col_mu[t], n)
    def unscale_std(t, s_flat):
        n = len(s_flat) // N_RHO
        return s_flat * np.tile(col_sd[t], n)

def metrics(model, level, y_true, y_mean, y_std, masks, model_time):
    """One dict per test subset. Headline metrics use the ACTIVE outputs only."""
    out = []
    for name, m in masks.items():
        yt, ym, ys = y_true[m], y_mean[m], y_std[m]
        err = ym - yt
        rmse, mae = np.sqrt((err ** 2).mean(0)), np.abs(err).mean(0)
        finite = bool(np.isfinite(ym).all() and np.isfinite(ys).all())   # diverged fit -> NaN metrics
        if finite:
            e, sig = err[:, ACTIVE], np.maximum(ys[:, ACTIVE], 1e-6)
            rel_l2 = float(np.linalg.norm(e) / np.linalg.norm(yt[:, ACTIVE]))
            nll = float((0.5 * np.log(2 * np.pi * sig ** 2) + e ** 2 / (2 * sig ** 2)).mean())
        else:
            rel_l2 = nll = float("nan")
        out.append({
            "model": model, "level": level, "subset": name, "n": int(m.sum()),
            "fit_diverged": (not finite),
            "rmse_mean": float(rmse[ACTIVE].mean()), "mae_mean": float(mae[ACTIVE].mean()),
            "rmse_minor": float(rmse[ACTIVE & MINOR].mean()), "rmse_major": float(rmse[ACTIVE & ~MINOR].mean()),
            "mae_minor": float(mae[ACTIVE & MINOR].mean()), "mae_major": float(mae[ACTIVE & ~MINOR].mean()),
            "rel_l2": rel_l2, "nll_mean": nll,
            "n_outputs_scored": int(ACTIVE.sum()),
            "per_output": {c: {"rmse": float(rmse[j]), "mae": float(mae[j])}
                           for j, c in enumerate(targ_cols)},
            "model_time_s": round(float(model_time), 2),
        })
    return out

def save_pred_csv(path, index, dist, mean, std):
    df = pd.DataFrame({"index": index, "nn_dist": dist})
    for j, c in enumerate(targ_cols):
        df[c] = mean[:, j]
    for j, c in enumerate(targ_cols):
        df[c + "_std"] = std[:, j]
    df.to_csv(path, index=False)

TABLE_COLS = ("model", "level", "subset", "n", "rmse_mean", "mae_mean", "rmse_minor", "rmse_major")

In [ ]:
import GPy
import GPy.util.linalg as _gpy_linalg
_gpy_linalg.force_F_ordered = np.asfortranarray   # same result as GPy's helper, without its debug print
from emukit.multi_fidelity.convert_lists_to_array import convert_x_list_to_array
from emukit.multi_fidelity.kernels import LinearMultiFidelityKernel
from emukit.multi_fidelity.models import GPyLinearMultiFidelityModel
from emukit.model_wrappers.gpy_model_wrappers import GPyMultiOutputWrapper

D = XsL_lf_tr.shape[1]
X_mf_train = convert_x_list_to_array([XsL_lf_tr, XsL_hf_tr])
X_mf_lf_te = np.hstack([XsL_lf_te, np.zeros((len(XsL_lf_te), 1))])
X_mf_hf_te = np.hstack([XsL_hf_te, np.ones((len(XsL_hf_te), 1))])

N_RESTARTS = 5                 # optimisation restarts for every GP (first restart = initial values)
HYP_LO, HYP_HI = 1e-6, 1e6     # bounds on every kernel variance, lengthscale and noise variance

def make_base_kernel():
    # Linear(GSH) x Matern52(rho) + Matern52(rho) + Matern52(all) + Bias
    return (GPy.kern.Linear(D - 1, active_dims=list(range(D - 1)))
            * GPy.kern.Matern52(1, active_dims=[D - 1], lengthscale=1.0)
            + GPy.kern.Matern52(1, active_dims=[D - 1], lengthscale=1.0)
            + GPy.kern.Matern52(D, active_dims=list(range(D)), lengthscale=3.0)
            + GPy.kern.Bias(D))

def make_delta_kernel():
    # fidelity discrepancy: smooth function of rho plus a constant
    return GPy.kern.Matern52(1, active_dims=[D - 1], lengthscale=1.0) + GPy.kern.Bias(D)

def make_mf_kernel():
    if KERNEL_SET == "standard":
        return LinearMultiFidelityKernel([GPy.kern.Matern52(D, ARD=True), GPy.kern.Matern52(D, ARD=True)])
    return LinearMultiFidelityKernel([make_base_kernel(), make_delta_kernel()])

def bound_hyperparameters(model):
    for p in model.flattened_parameters:
        if p.hierarchy_name().split(".")[-1] == "scale" or p.is_fixed:   # AR1 rho ("scale") stays free
            continue
        p.constrain_bounded(HYP_LO, HYP_HI, warning=False)

def fit_predict_ar1(t):
    y = np.concatenate([scale_y(t, yL_lf_tr[t]), scale_y(t, yL_hf_tr[t])])[:, None]
    model = GPyLinearMultiFidelityModel(X_mf_train, y, make_mf_kernel(), n_fidelities=2)
    if KERNEL_SET != "standard":
        model.mixed_noise.Gaussian_noise.variance = 0.2
        model.mixed_noise.Gaussian_noise_1.variance = 0.2
    bound_hyperparameters(model)
    wrap = GPyMultiOutputWrapper(model, 2, n_optimization_restarts=N_RESTARTS, verbose_optimization=False)
    wrap.optimize()
    m_lf, v_lf = wrap.predict(X_mf_lf_te)
    m_hf, v_hf = wrap.predict(X_mf_hf_te)
    return (unscale_mean(t, m_lf.ravel()), unscale_std(t, np.sqrt(np.maximum(v_lf.ravel(), 0))),
            unscale_mean(t, m_hf.ravel()), unscale_std(t, np.sqrt(np.maximum(v_hf.ravel(), 0))))

np.random.seed(SEED)
t0 = time.time()
pred = {}
for t in TARGETS:
    tj = time.time()
    pred[t] = fit_predict_ar1(t)
    print(f"AR1 [{t:<5s}] {time.time()-tj:6.1f}s", flush=True)
ar1_time = time.time() - t0

ar1_lf_mean = long_to_wide({t: pred[t][0] for t in TARGETS}, N_LF_EVAL)
ar1_lf_std  = long_to_wide({t: pred[t][1] for t in TARGETS}, N_LF_EVAL)
ar1_hf_mean = long_to_wide({t: pred[t][2] for t in TARGETS}, N_HF_TEST)
ar1_hf_std  = long_to_wide({t: pred[t][3] for t in TARGETS}, N_HF_TEST)

In [ ]:
from emukit.multi_fidelity.models.non_linear_multi_fidelity_model import (
    make_non_linear_kernels, NonLinearMultiFidelityModel)

def fit_predict_nargp(t):
    y = np.concatenate([scale_y(t, yL_lf_tr[t]), scale_y(t, yL_hf_tr[t])])[:, None]
    if KERNEL_SET == "standard":
        kernels = make_non_linear_kernels(GPy.kern.Matern52, 2, D, ARD=True)
    else:
        # level 2 sees [x, f_LF(x)] (column D); corrections restricted to rho
        kern2 = (GPy.kern.Matern52(1, active_dims=[D - 1], lengthscale=1.0)
                 * GPy.kern.Matern52(1, active_dims=[D])
                 + GPy.kern.Matern52(1, active_dims=[D - 1], lengthscale=1.0) + GPy.kern.Bias(D))
        kernels = [make_base_kernel(), kern2]
    model = NonLinearMultiFidelityModel(X_mf_train, y, n_fidelities=2, kernels=kernels,
                                        verbose=False, optimization_restarts=N_RESTARTS)

    def fit_level(m):
        # NARGP reference protocol: noise at 1% of the level's target variance, fixed for a
        # warm-start optimisation, then released and re-optimised with restarts
        bound_hyperparameters(m)
        m.Gaussian_noise.variance.unfix()
        m.Gaussian_noise.variance.constrain_bounded(HYP_LO, HYP_HI, warning=False)
        m.Gaussian_noise.variance = 0.01 * float(m.Y.var())
        m.Gaussian_noise.variance.fix(warning=False)
        m.optimize(max_iters=500)
        m.Gaussian_noise.variance.unfix()
        m.Gaussian_noise.variance.constrain_bounded(HYP_LO, HYP_HI, warning=False)
        m.optimize_restarts(N_RESTARTS, verbose=False, robust=True)

    # same sequence as emukit's NonLinearMultiFidelityModel.optimize(): level 1, rebuild the
    # level-2 input from the optimised level-1 posterior mean, then level 2
    fit_level(model.models[0])
    is_hf = model.X[:, -1] == 1
    prev_mean, _ = model._predict_deterministic(model.X[is_hf, :-1], 1)
    model.models[1].set_X(np.concatenate([model.models[1].X[:, :-1], prev_mean], axis=1))
    fit_level(model.models[1])
    m_lf, v_lf = model.predict(X_mf_lf_te)
    m_hf, v_hf = model.predict(X_mf_hf_te)
    return (unscale_mean(t, m_lf.ravel()), unscale_std(t, np.sqrt(np.maximum(v_lf.ravel(), 0))),
            unscale_mean(t, m_hf.ravel()), unscale_std(t, np.sqrt(np.maximum(v_hf.ravel(), 0))))

np.random.seed(SEED)
t0 = time.time()
pred = {}
for t in TARGETS:
    tj = time.time()
    pred[t] = fit_predict_nargp(t)
    print(f"NARGP [{t:<5s}] {time.time()-tj:6.1f}s", flush=True)
nargp_time = time.time() - t0

nargp_lf_mean = long_to_wide({t: pred[t][0] for t in TARGETS}, N_LF_EVAL)
nargp_lf_std  = long_to_wide({t: pred[t][1] for t in TARGETS}, N_LF_EVAL)
nargp_hf_mean = long_to_wide({t: pred[t][2] for t in TARGETS}, N_HF_TEST)
nargp_hf_std  = long_to_wide({t: pred[t][3] for t in TARGETS}, N_HF_TEST)

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

summaries = (metrics("AR1",   "LF_test", Y_lf_te, ar1_lf_mean,   ar1_lf_std,   lf_masks, ar1_time)
           + metrics("AR1",   "HF_test", Y_hf_te, ar1_hf_mean,   ar1_hf_std,   hf_masks, ar1_time)
           + metrics("NARGP", "LF_test", Y_lf_te, nargp_lf_mean, nargp_lf_std, lf_masks, nargp_time)
           + metrics("NARGP", "HF_test", Y_hf_te, nargp_hf_mean, nargp_hf_std, hf_masks, nargp_time))

table = pd.DataFrame([{k: s[k] for k in TABLE_COLS} for s in summaries])
print(table.to_string(index=False))
table.to_csv(OUT_DIR / "ar1_nargp_results.csv", index=False)

save_pred_csv(OUT_DIR / "pred_ar1_lf_test.csv",   lf_te_index, lf_te_dist, ar1_lf_mean,   ar1_lf_std)
save_pred_csv(OUT_DIR / "pred_ar1_hf_test.csv",   hf_te_index, hf_te_dist, ar1_hf_mean,   ar1_hf_std)
save_pred_csv(OUT_DIR / "pred_nargp_lf_test.csv", lf_te_index, lf_te_dist, nargp_lf_mean, nargp_lf_std)
save_pred_csv(OUT_DIR / "pred_nargp_hf_test.csv", hf_te_index, hf_te_dist, nargp_hf_mean, nargp_hf_std)

with open(OUT_DIR / "ar1_nargp_results.json", "w") as f:
    json.dump({"config": CONFIG, "summaries": summaries}, f, indent=2)
print("saved to", OUT_DIR)

In [ ]:
# FLD curves (minor vs major strain) of the first HF test textures: true vs predicted
n_show = 4
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4), sharey=True)
for k, ax in enumerate(axes):
    for y_row, label, col in ((Y_hf_te[k], "HF true", "k"), (ar1_hf_mean[k], "AR1", "tab:blue"),
                              (nargp_hf_mean[k], "NARGP", "tab:orange")):
        ax.plot(y_row[0::2], y_row[1::2], marker="o", ms=4, label=label, color=col)
    ax.set_title(f"HF test index={hf_te_index[k]}")
    ax.set_xlabel("minor strain")
axes[0].set_ylabel("major strain"); axes[0].legend()
fig.suptitle(f"{TAG}: HF test FLD curves")
fig.tight_layout()
plt.show()